In [1]:
# --- Run ---
import datetime

run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

In [2]:
import subprocess, sys, os

"""
This output is:
 1. pubs.txt - Paper list for CWTS tool
    Columns: int_id  paper identifier for cwts, core_pub (always 1 as not using core feature, but it has to be in)

 2. cit_links.txt - Citation network edges
    Columns: 
    int_id1 - citing paper, 
    int_id2 - cited paper, 
    weight - citation strength 0-2 higher= stronger
    Note: Each edge appears twice (A→B and B→A) for undirected format
        paper 5 cites Paper 12  →  row: 5, 12, 0.85
        Paper 12 cites Paper 5  →  row: 12, 5, 0.85  (same edge, reversed)

 3. pub_metadata.txt - Paper details lookup table
    Columns: int_id, pub_id, is_frontiers, journal, date, title
    - int_id: sequential CWTS ID (joins to classification.txt)
    - pub_id: airak PublicationId (joins to BigQuery tables)
 JOIN KEY: int_id links all files together, this is cwts identifier
"""
print("ere")
env = os.environ.copy()
print("ere")
env["START_YEAR"] = "2026"
env["END_YEAR"] = "2026"
env["NETWORK_MODE"] = "global"
env["run_timestamp"] = run_timestamp
print("ere")
result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)
print("ere")
print(result.stdout)
print(result.stderr)
print("last")

ere
ere
ere
ere

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026-06-16 14:34:28,262 [INFO] ============================================================
2026-06-16 14:34:28,262 [INFO] Configuration
2026-06-16 14:34:28,262 [INFO] ============================================================
2026-06-16 14:34:28,263 [INFO]   BQ_PROJECT              : ocean-tech-adv-analytics-p-usr
2026-06-16 14:34:28,263 [INFO]   AIRAK_DATASET           : ocean-breeze-tier-1.airak
2026-06-16 14:34:28,263 [INFO]   NETWORK_MODE            : global
2026-06-16 14:34:28,263 [INFO]   START_YEAR              : 2026
2026-06-16 14:34:28,263 [INFO]   END_YEAR                : 2026
2026-06-16 14:34:28,264 [INFO]   TOP_N_JOURNALS          : 5
2026-06-16 14:34:28,264 [INFO]   JOURNAL_IDS_OVERRIDE    : (none)
2026-06-16 14:34:2

In [3]:
import subprocess
import datetime
import os
import pandas as pd

# --- Parameters ---
params = {
    "largest_component_only": "true",
    "iterations": "1000",
    "micro_resolution": "2e-4",
    "micro_min_cluster_size": "10000",
    "meso_resolution": "4.9e-7",
    "meso_min_cluster_size": "10000",
    "macro_resolution": "2.2e-8",
    "macro_min_cluster_size": "200000",
}

input_files = {
    "pubs": "cwts_output/pubs.txt",
    "cit_links": "cwts_output/cit_links.txt",
    "output": "cwts_output/classification.txt",
    "jar": "publicationclassification.jar",
}


result = subprocess.run(
    [
        "java",
        "-cp",
        input_files["jar"],
        "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
        input_files["pubs"],
        input_files["cit_links"],
        input_files["output"],
        params["largest_component_only"],
        params["iterations"],
        params["micro_resolution"],
        params["micro_min_cluster_size"],
        params["meso_resolution"],
        params["meso_min_cluster_size"],
        params["macro_resolution"],
        params["macro_min_cluster_size"],
    ],
    capture_output=True,
    text=True,
)

# --- Log ---
os.makedirs("logs", exist_ok=True)
log_path = f"logs/cwts_run_{run_timestamp}.log"

with open(log_path, "w") as f:
    f.write(f"CWTS Publication Classification Run\n")
    f.write(f"{'='*50}\n")
    f.write(f"Timestamp : {run_timestamp}\n\n")

    f.write(f"Input Files\n{'-'*30}\n")
    for k, v in input_files.items():
        f.write(f"  {k:<20}: {v}\n")

    f.write(f"\nParameters\n{'-'*30}\n")
    for k, v in params.items():
        f.write(f"  {k:<26}: {v}\n")

    f.write(f"\nReturn Code: {result.returncode}\n")

    f.write(f"\nSTDOUT\n{'-'*30}\n")
    f.write(result.stdout or "(empty)\n")

    f.write(f"\nSTDERR\n{'-'*30}\n")
    f.write(result.stderr or "(empty)\n")

print(f"Log written to: {log_path}")
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Load classification.txt
classification = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["int_id", "micro", "meso", "macro"],
)

# Upload to BigQuery
BQ_DEST_PROJECT = "ocean-tech-adv-analytics-c-tfs"
BQ_DEST_DATASET = "scope_drift_raw"


# classification.to_gbq(
#     f"{dataset}.classification_raw_{run_timestamp}",
#     project_id=project,
#     if_exists="replace",
# )

# Upload to BigQuery

classification.to_gbq(
    f"{BQ_DEST_DATASET}.classification_raw_{run_timestamp}",
    project_id=BQ_DEST_PROJECT,
    if_exists="replace",
)
print(f"  → BigQuery: {BQ_DEST_DATASET}.classification_raw_{run_timestamp}")

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Log written to: logs/cwts_run_20260616_143424.log
PublicationClassificationCreator version 1.1.0
By Nees Jan van Eck
Centre for Science and Technology Studies (CWTS), Leiden University

Reading citation network from file... Finished!
Reading citation network from file took 0h 0m 0s.
Citation network:
	Number of publications: 549424
	Number of citation links: 528729
	Total publication weight: 549424
	Total citation link weight: 512090

Identifying largest connected component in citation network... Finished!
Identifying largest connected component in citation network took 0h 0m 0s.
Largest connected component:
	Number of publications: 268072
	Number of citation links: 342620
	Total publication weight: 268072
	Total citation link weight: 333338

Creating publication classification...
	Clustering algorithm: Leiden algorithm
	Number of iterations: 1000
	Random seed: 0

Adding micro-level classification...
Creating clustering... Finished! 2177 clusters created.
Reassigning small clusters... 

C:\Users\sophie.wilson\AppData\Local\Temp\ipykernel_103096\1444878952.py:99: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  classification.to_gbq(
100%|██████████| 1/1 [00:00<?, ?it/s]

  → BigQuery: scope_drift_raw.classification_raw_20260616_143424


In [ ]:
import pandas as pd



df = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)



print(f"Total classified: {len(df):,}")


for level in ["micro", "meso", "macro"]:


    vc = df[level].value_counts()


    print(f"\n{level.upper()}: {len(vc):,} clusters")


    print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")


    print(f"  Smallest: {vc.iloc[-1]:,}")


    print(f"  Median  : {vc.median():.0f}")

Total classified: 268,072

MICRO: 14 clusters
  Largest : 37,484 (14.0%)
  Smallest: 10,670
  Median  : 15938

MESO: 3 clusters
  Largest : 168,696 (62.9%)
  Smallest: 28,550
  Median  : 70826

MACRO: 1 clusters
  Largest : 268,072 (100.0%)
  Smallest: 268,072
  Median  : 268072


### Labelling with GPT

In [5]:
import label_clusters

# Run the script
label_clusters.main()

Loading classification...
Loading metadata...
Loading citation links...
Calculating citation counts...
Merged: 268,072 publications

--- Labelling macro (1 clusters) ---
  [1/1] cluster 0 (0 papers) → No Papers

Saved → cwts_output\macro_labels.csv

Done.


In [6]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("cwts_output")
classif_path = OUTPUT_DIR / "classification.txt"
titles_path = OUTPUT_DIR / "pub_titles.txt"
classif = pd.read_csv(
    classif_path,
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)

print("Loading titles...")
titles = pd.read_csv(
    titles_path,
    sep="\t",
    header=None,
    names=["pub_no", "title"],
)

merged = classif.merge(titles, on="pub_no")
print(f"Merged: {len(merged):,} publications")

# merged.sample(5000).to_csv('full_merged.csv')

Loading titles...


FileNotFoundError: [Errno 2] No such file or directory: 'cwts_output\\pub_titles.txt'

In [ ]:
merged.sample(5000).to_csv("merged_sample.csv")

In [ ]:
pd.set_option("display.max_colwidth", None)
merged[["macro", "title"]].sample(500)

### looking at scope

In [ ]:
import pandas as pd
from pathlib import Path

core = pd.read_csv(
    "cwts_output/frontiers_core.txt",
    sep="\t",
    header=None,
    names=["pub_no", "is_frontiers", "journal"],
)

print(core["journal"].unique())

In [ ]:
import importlib
import journal_scope

# Override config variables
journal_scope.SCOPE_LEVEL = "macro"
journal_scope.SCOPE_THRESHOLD = 0.80
journal_scope.MIN_PAPERS = 50
journal_scope.USE_GPT = False
journal_scope.OUTPUT_DIR = Path("cwts_output")

journal_scope.TARGET_JOURNALS = [
    "Frontiers in Immunology",
    "Frontiers in Public Health",
    "Frontiers in Medicine",
    "Frontiers in Oncology",
    "Frontiers in Psychology",
]

# Run
journal_scope.main()

In [ ]:
pd.set_option("display.max_colwidth", 50)
pd.read_csv("cwts_output/journal_scope.csv")